In [23]:
import pandas as pd

df = pd.read_csv(r"C:\Users\hamza\Downloads\teaching_prices_messy (1).csv")

print("Shape:", df.shape)
print(df.head())
print(df.dtypes)
print(df["item"].value_counts(dropna=False))

Shape: (97, 6)
         date      item    city  price                       unit  \
0  2026-01-01  Staple A  City 1  101.0  PKR per kg (illustrative)   
1  2026-01-01  Staple A  City 2  107.0  PKR per kg (illustrative)   
2  2026-01-01     Oil B  City 1  298.0  PKR per kg (illustrative)   
3  2026-01-01     Oil B  City 2  304.0  PKR per kg (illustrative)   
4  2026-01-01   Pulse C  City 1  200.0  PKR per kg (illustrative)   

                                          data_label  
0  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
1  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
2  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
3  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
4  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
date          str
item          str
city          str
price         str
unit          str
data_label    str
dtype: object
item
Staple A       24
Oil B          24
Pulse C        24
Sweetener D    24
oil b           1
Name: count, dtype: int

In [24]:
# price is text because of "1,250" with a comma
df["price"] = df["price"].astype(str).str.replace(",", "", regex=False)
df["price"] = pd.to_numeric(df["price"], errors="coerce")

print(df["price"].isna().sum(), "missing prices")
print(df[df["price"].isna()])

1 missing prices
         date     item    city  price                       unit  \
5  2026-01-01  Pulse C  City 2    NaN  PKR per kg (illustrative)   

                                          data_label  
5  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  


In [25]:
# 1. Duplicates
print("Duplicates:")
print(df[df.duplicated(keep=False)])

# 2. Mislabeled item - lowercase 'oil b'
print("\nLowercase item:")
print(df[df["item"] == "oil b"])

# 3. Extreme values
print("\nExtreme values:")
print(df[df["price"] > 1000])

Duplicates:
          date      item    city  price                       unit  \
8   2026-01-08  Staple A  City 1   99.5  PKR per kg (illustrative)   
96  2026-01-08  Staple A  City 1   99.5  PKR per kg (illustrative)   

                                           data_label  
8   ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
96  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  

Lowercase item:
          date   item    city  price                       unit  \
40  2026-02-05  oil b  City 1  108.5  PKR per kg (illustrative)   

                                           data_label  
40  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  

Extreme values:
          date         item    city   price                       unit  \
17  2026-01-15     Staple A  City 2  1250.0  PKR per kg (illustrative)   
30  2026-01-22  Sweetener D  City 1  9999.0  PKR per kg (illustrative)   

                                           data_label  
17  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS O

In [26]:
df_clean = df.copy()

# 1. Drop 1 duplicate: Staple A, City 1, 2026-01-08, 99.5
df_clean = df_clean.drop_duplicates()

# 2. Fix mislabel: 'oil b' 108.5 is Staple A range, not Oil B range
df_clean.loc[df_clean["item"] == "oil b", "item"] = "Staple A"

# 3. Drop flagged / implausible values - do NOT guess them
# Staple A 1,250 and Sweetener D 9,999
flagged = df_clean[df_clean["price"].isin([1250, 9999])]
print("Flagged rows dropped:")
print(flagged)

df_clean = df_clean[~df_clean["price"].isin([1250, 9999])]

# 4. Keep Pulse C NaN as missing - do NOT fill with 0
print("\nRows after cleaning:", len(df_clean))
print("Valid prices per item:")
print(df_clean.groupby("item")["price"].count())

Flagged rows dropped:
          date         item    city   price                       unit  \
17  2026-01-15     Staple A  City 2  1250.0  PKR per kg (illustrative)   
30  2026-01-22  Sweetener D  City 1  9999.0  PKR per kg (illustrative)   

                                           data_label  
17  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  
30  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...  

Rows after cleaning: 94
Valid prices per item:
item
Oil B          24
Pulse C        23
Staple A       23
Sweetener D    23
Name: price, dtype: int64


In [27]:
rows = []

for item, g in df_clean.groupby("item"):
    valid = g["price"].dropna()
    first_wk = g[g["date"] == g["date"].min()]["price"].mean()
    last_wk = g[g["date"] == g["date"].max()]["price"].mean()
    pct = (last_wk - first_wk) / first_wk * 100

    rows.append({
        "Item": item,
        "Valid n": len(valid),
        "Mean": round(valid.mean(), 2),
        "Median": round(valid.median(), 2),
        "First wk avg": round(first_wk, 2),
        "Last wk avg": round(last_wk, 2),
        "First-to-last %": round(pct, 2)
    })

table1 = pd.DataFrame(rows)
print(table1.to_string(index=False))

       Item  Valid n   Mean  Median  First wk avg  Last wk avg  First-to-last %
      Oil B       24 324.83   324.0         301.0        347.0            15.28
    Pulse C       23 205.91   206.0         200.0        210.5             5.25
   Staple A       23 111.26   112.5         104.0        117.5            12.98
Sweetener D       23 167.15   167.5         152.0        181.5            19.41


In [28]:
# Use df BEFORE dropping flagged values, but AFTER duplicate + mislabel fix
# Rebuild it for clarity:
df_before_flag = df.drop_duplicates().copy()
df_before_flag.loc[df_before_flag["item"] == "oil b", "item"] = "Staple A"

for item in ["Staple A", "Sweetener D"]:
    with_flag = df_before_flag[df_before_flag["item"] == item]["price"].dropna()
    without_flag = df_clean[df_clean["item"] == item]["price"].dropna()

    print(f"\n{item}")
    print(f"With flagged:    n={len(with_flag)}, mean={with_flag.mean():.2f}, median={with_flag.median():.2f}")
    print(f"Without flagged: n={len(without_flag)}, mean={without_flag.mean():.2f}, median={without_flag.median():.2f}")
    print(f"Mean change: {with_flag.mean() - without_flag.mean():.2f}")


Staple A
With flagged:    n=24, mean=158.71, median=112.50
Without flagged: n=23, mean=111.26, median=112.50
Mean change: 47.45

Sweetener D
With flagged:    n=24, mean=576.81, median=167.75
Without flagged: n=23, mean=167.15, median=167.50
Mean change: 409.66


In [29]:
table1["Mean - Median"] = (table1["Mean"] - table1["Median"]).round(2)
print(table1[["Item", "Mean", "Median", "Mean - Median"]].to_string(index=False))

       Item   Mean  Median  Mean - Median
      Oil B 324.83   324.0           0.83
    Pulse C 205.91   206.0          -0.09
   Staple A 111.26   112.5          -1.24
Sweetener D 167.15   167.5          -0.35
